# BAT 157-month radio-quiet AGN sensitivity

For each BAT-selected source and cutoff, use `SourceInjector` with
`fluctuate=False` to generate expected source counts and add the expected
background counts to form one Asimov dataset. Fit their
normalizations with threeML/MINUIT to calculate the Asimov detection TS.

- Read the BAT catalog directly. Include radio-quiet Seyferts classified RQ
  in `RADIO_CLASS_PATH`, with BAT 14–195 keV flux
  ≥ `MIN_BAT_FLUX_ERG_CM2_S`; there is no top-50 or COSI preselection.
- Keep a measured central index/cutoff from `AGN_Index_ecut.txt`. Otherwise use
  the BAT index and scan 100–1000 keV. NGC 4151 follows this scan.
- Preserve the catalog BAT flux at every cutoff. Only the source and background
  normalizations vary in the fit; position, index, and cutoff stay fixed.
- Exclude zero-background template cells from both components everywhere.
- Evaluate each source separately against the background. Stop its cutoff scan
  at the first valid three-month FOV-estimated Asimov TS ≥ 25. Do not skip other sources.
  
The outputs are tables of detectable sources, sources without a detection,
and sources with measured Baloković cutoffs below 100 keV. The final CSV
contains every selected radio-quiet source.

In [20]:
import os
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = "1"

import logging
import re
import warnings
from pathlib import Path

import astropy.units as u
from astropy.coordinates import SkyCoord
import h5py
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.integrate import quad
from astromodels import Cutoff_powerlaw, Model, PointSource, Parameter
from astromodels.utils.configuration import astromodels_config
from threeML import DataList, JointLikelihood
from threeML.io.logging import quiet_mode
from threeML.minimizer.minimization import FitFailed
from histpy import Histogram

from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.data_io import EmCDSBinnedData
from cosipy.response import FullDetectorResponse, BinnedInstrumentResponse
from cosipy.response.functions import get_integrated_spectral_model
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.source_injector import SourceInjector

logging.getLogger("cosipy").setLevel(logging.WARNING)
quiet_mode()

In [21]:
# Set the two input directories; save the final CSV in the current directory.
DC4_DIR = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files")
AGN_DIR = Path("/Users/parshadkp/Software/AGN-HE-detectability")

CATALOG_PATH = AGN_DIR / "BAT_157m_catalog_20250525.csv"
BALOKOVIC_PATH = AGN_DIR / "AGN_Index_ecut.txt"
RADIO_CLASS_PATH = AGN_DIR / "phillipson_2023_table1_radio_classes.csv"
paths = {
    "orientation": DC4_DIR / "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits",
    "response": DC4_DIR / (
        "ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284."
        "filtered.nonsparse.binnedimaging.imagingresponse.h5"
    ),
    "background": DC4_DIR / "gal_DC4_bkg_continuum_binned_smoothed_fwhm_10.0deg.hdf5",
}
TOTAL_BACKGROUND_PATH = (DC4_DIR / "Background" /
    "Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck.hdf5")
CHECKPOINT_PATH = Path.cwd() / "bat157_asimov_results.csv"

## Choose the files and analysis settings

The default minimum flux, MIN_BAT_FLUX_ERG_CM2_S = **5 × 10⁻¹¹ erg cm⁻² s⁻¹**.

Each cutoff uses one deterministic Asimov dataset.
Set `USE_MEASURED_CUTOFFS = False` to scan every selected source with its BAT index.

In [22]:
# Prior scans suggest a limit near 6e-11; include a margin below it.
MIN_BAT_FLUX_ERG_CM2_S = 5e-11
BAT_CLASSES = (4, 5)
USE_MEASURED_CUTOFFS = True

SOURCE_NAMES = None                 # Optional subset after the flux and radio-quiet selection.
CUTOFF_VALUES_KEV = np.arange(100.0, 1001.0, 100.0)
TARGET_TS = 25.0
FOV_MAX_OFFAXIS_DEG = 60.0
EXPOSURE_SCALE_2YR = 8.0             # 24 months / 3 months.

BAT_BAND_KEV = (14.0, 195.0)
ENERGY_BAND_KEV = (100.0, 10000.0)
KEV_TO_ERG = 1.602176634e-9

## Read the radio-quiet source catalog

Apply the BAT class and flux selection, then retain only sources classified RQ
in the radio CSV. Published names and explicit catalog aliases are matched;
radio-loud and unclassified sources are excluded. Keep the full BAT catalog rank
to break ties in the result tables. Photon index Γ is positive in the output:
`photon_index = -astromodel_index`.

In [23]:
def band_fluxes(shape, band):
    photon = quad(lambda energy: float(shape(energy)), *band,
                  epsabs=1e-30, epsrel=1e-9)[0]
    energy = quad(lambda energy: energy * float(shape(energy)), *band,
                  epsabs=1e-30, epsrel=1e-9)[0]
    return photon, energy * KEV_TO_ERG


def spectrum_at_cutoff(index, cutoff, bat_flux):
    """Return a cutoff spectrum with the reference BAT energy flux."""
    integral = quad(lambda energy: energy ** (index + 1) * np.exp(-energy / cutoff),
                    *BAT_BAND_KEV, epsabs=1e-30, epsrel=1e-11)[0]
    amplitude = bat_flux / (KEV_TO_ERG * integral)
    shape = Cutoff_powerlaw()
    shape.set_units(u.keV, u.Unit("1 / (cm2 s keV)"))
    shape.K.max_value = max(shape.K.max_value, 10 * amplitude)
    shape.K.value = amplitude
    shape.index.value = index
    shape.piv.value = 1.0
    shape.xc.value = cutoff
    return shape

Match literature names without spaces or punctuation. Retain published cutoff
values and limits for the output tables. Only central measurements fix the
spectrum; a lower limit is handled by the cutoff scan.

Use the radio-classification CSV to select only radio-quiet AGNs before
building their cutoff scenarios. Match published names and explicit catalog
aliases; exclude radio-loud and unclassified sources.


In [ ]:
def source_key(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())


def source_name(label):
    name = re.sub(r"[^A-Za-z0-9]+", "_", label.strip()).strip("_")
    return "BAT_" + name if name[0].isdigit() else name


def read_balokovic_cutoffs(path):
    """Read fit-ready measurements and published cutoff labels, including limits."""
    measurements, published_cutoffs = {}, {}
    for line in path.read_text().splitlines():
        fields = [field.strip() for field in line.split("\t")]
        if len(fields) < 5 or not re.match(r"\d", fields[1]):
            continue
        gamma_text, cutoff_text = fields[3:5]
        gamma, cutoff = [float(re.search(r"[-+]?\d*\.?\d+", text).group())
                         for text in (gamma_text, cutoff_text)]
        key = source_key(fields[0])
        if key == "cyga":
            key = "cygnusa"
        limit = ""
        if ">" in cutoff_text or r"\gt" in cutoff_text:
            limit = ">"
        elif "<" in cutoff_text or r"\lt" in cutoff_text:
            limit = "<"
        published_cutoffs[key] = f"{limit}{cutoff:g}"
        if not limit and not any(token in gamma_text for token in (">", "<", r"\gt", r"\lt")):
            measurements[key] = (gamma, cutoff)
    if not published_cutoffs:
        raise ValueError("No cutoffs found in the literature table")
    return measurements, published_cutoffs

In [25]:
radio_table = pd.read_csv(RADIO_CLASS_PATH)
matched_radio = radio_table.dropna(subset=["cosipy_source"])
if not matched_radio["cosipy_source"].is_unique or not radio_table["radio_class"].isin(["RQ", "RL"]).all():
    raise ValueError("Radio classifications need unique source names and RQ/RL values")

radio_class = {}
for row in radio_table.to_dict("records"):
    classification = {"RQ": "radio quiet", "RL": "radio loud"}[row["radio_class"]]
    # Match both the published name and any explicit catalog alias.
    radio_class[source_name(row["paper_source"])] = classification
    if pd.notna(row["cosipy_source"]):
        radio_class[row["cosipy_source"]] = classification

In [26]:
catalog = pd.read_csv(CATALOG_PATH, skiprows=[1], skipinitialspace=True)
catalog.columns = catalog.columns.str.strip()
for column in ("CL2", "FLUX", "GAMMA", "CTPT_RA", "CTPT_DEC"):
    catalog[column] = pd.to_numeric(catalog[column], errors="coerce")
catalog = catalog.loc[catalog["CL2"].isin(BAT_CLASSES) & (catalog["FLUX"] > 0)].copy()
catalog = catalog.sort_values(["FLUX", "BAT_NAME"], ascending=[False, True])
catalog["catalog_rank"] = np.arange(1, len(catalog) + 1)
catalog["bat_14_195_energy_flux_erg_cm2_s"] = catalog["FLUX"] * 1e-12
selected_catalog = catalog.loc[
    catalog["bat_14_195_energy_flux_erg_cm2_s"] >= MIN_BAT_FLUX_ERG_CM2_S
].copy()
selected_catalog["counterpart"] = selected_catalog["COUNTERPART_NAME"].str.strip()
selected_catalog["source"] = selected_catalog["counterpart"].map(source_name)
selected_catalog["radio_class"] = selected_catalog["source"].map(radio_class).fillna("unclassified")
selected_catalog = selected_catalog.loc[selected_catalog["radio_class"] == "radio quiet"].copy()
if SOURCE_NAMES is not None:
    unknown = set(SOURCE_NAMES) - set(selected_catalog["source"])
    if unknown:
        raise ValueError(f"Sources outside the flux/radio-quiet selection: {sorted(unknown)}")
    selected_catalog = selected_catalog.loc[selected_catalog["source"].isin(SOURCE_NAMES)]
if selected_catalog.empty or selected_catalog["source"].duplicated().any():
    raise ValueError("Select at least one radio-quiet source, with distinct counterpart names")
coordinates = selected_catalog[["CTPT_RA", "CTPT_DEC"]].to_numpy()
if (not np.isfinite(coordinates).all() or np.any((coordinates[:, 0] < 0) | (coordinates[:, 0] >= 360))
        or np.any(np.abs(coordinates[:, 1]) > 90)):
    raise ValueError("Selected sources need valid counterpart coordinates")

Build the list of possible source/cutoff combinations. A measured cutoff gives
one combination. Other sources get the ascending cutoff grid. Only the combinations
actually reached by the search will appear in `summary`.

In [ ]:
cutoff_measurements, published_cutoffs = read_balokovic_cutoffs(BALOKOVIC_PATH)
if not USE_MEASURED_CUTOFFS:
    cutoff_measurements = {}
scenario_rows = []
for row in selected_catalog.to_dict("records"):
    measurement = cutoff_measurements.get(source_key(row["counterpart"]))
    gamma, fixed_cutoff = measurement if measurement is not None else (row["GAMMA"], None)
    if not np.isfinite(gamma) or not 0 < gamma <= 10:
        raise ValueError(f"{row['counterpart']}: invalid photon index")
    cutoffs = [fixed_cutoff] if fixed_cutoff is not None else CUTOFF_VALUES_KEV
    for order, cutoff in enumerate(cutoffs):
        scenario_rows.append({
            "catalog_rank": row["catalog_rank"], "source": row["source"],
            "counterpart": row["counterpart"], "bat_name": row["BAT_NAME"].strip(),
            "ra_deg": row["CTPT_RA"], "dec_deg": row["CTPT_DEC"],
            "scenario_type": "balokovic_fixed" if measurement is not None else "bat_cutoff_scan",
            "scenario_order": order, "cutoff_keV": float(cutoff), "astromodel_index": -gamma,
            "bat_14_195_energy_flux_erg_cm2_s": row["bat_14_195_energy_flux_erg_cm2_s"],
        })
scenarios = pd.DataFrame(scenario_rows)
maximum_scenarios = len(scenarios)
print(f"{len(selected_catalog)} radio-quiet sources with BAT flux ≥ {MIN_BAT_FLUX_ERG_CM2_S:g} "
      f"erg cm⁻² s⁻¹; at most {maximum_scenarios:,} Asimov datasets.")

## Open the response, spacecraft history, and background

Check native bin alignment once at input.


In [28]:
def check_counts(values, label):
    if not np.isfinite(values).all() or np.any(values < 0) or values.sum() <= 0:
        raise ValueError(f"{label} must be finite, nonnegative, and nonzero")


sc_orientation = SpacecraftHistory.open(paths["orientation"])
background = Histogram.open(paths["background"]).project("Em", "Phi", "PsiChi").to_dense()
background_values = np.asarray(background.contents, dtype=float)
check_counts(background_values, "Background")
keep_cells = background_values > 0
background_counts = background_values[keep_cells]
livetime_s = float(sc_orientation.cumulative_livetime().to_value(u.s))
if not np.isfinite(livetime_s) or livetime_s <= 0:
    raise ValueError("Spacecraft history must have positive finite livetime")
background_rate_hz = background_counts.sum() / livetime_s

with FullDetectorResponse.open(paths["response"]) as detector_response:
    for axis in (detector_response.axes["Ei"], detector_response.measurement_axes["Em"],
                 background.axes["Em"]):
        edges = axis.edges.to_value(u.keV)
        if not np.all(np.diff(edges) > 0) or not np.allclose(edges[[0, -1]], ENERGY_BAND_KEV):
            raise ValueError("Response and background must span 100–10,000 keV")
    for label in ("Em", "Phi"):
        if background.axes[label] != detector_response.measurement_axes[label]:
            raise ValueError(f"Background/response {label} axes differ")
    if "Pol" in detector_response.axes.labels:
        raise ValueError("Use the unpolarized DC4 continuum response")

In [29]:
def load_background_time_energy(path):
    """Project sparse Time/Em/Phi/PsiChi counts after removing empty-template cells."""
    with h5py.File(path, "r") as file:
        axes = file["hist/axes"]
        if [axes[str(i)].attrs["label"] for i in range(4)] != ["Time", "Em", "Phi", "PsiChi"]:
            raise ValueError("Unexpected time-resolved background axes")
        for i, label in enumerate(("Em", "Phi", "PsiChi"), start=1):
            np.testing.assert_array_equal(axes[str(i)][...], np.asarray(background.axes[label].edges))
        for key in ("coordsys", "nside", "scheme"):
            if axes["3"].attrs[key] != file_smooth_psichi_attributes[key]:
                raise ValueError(f"Background PsiChi {key} differs")
        if file["hist/contents/fill_value"][()] != 0:
            raise ValueError("Expected a zero-filled sparse background")
        shape = tuple(file["hist/contents/shape"][...])
        counts = np.zeros(shape[:2])
        coords, values = file["hist/contents/coords"], file["hist/contents/data"]
        for start in range(0, len(values), 2_000_000):
            indices = coords[:, start:start + 2_000_000]
            weights = values[start:start + 2_000_000]
            retained = keep_cells[indices[1], indices[2], indices[3]]
            counts += np.bincount(
                indices[0, retained] * shape[1] + indices[1, retained],
                weights=weights[retained], minlength=counts.size,
            ).reshape(counts.shape)
        time_edges = axes["0"][...]
    check_counts(counts, "Time-resolved background")
    return time_edges, counts


with h5py.File(paths["background"], "r") as file:
    file_smooth_psichi_attributes = dict(file["hist/axes/3"].attrs)
time_edges, counts_te = load_background_time_energy(TOTAL_BACKGROUND_PATH)
print(f"Retained {keep_cells.sum():,}/{keep_cells.size:,} cells; "
      f"background rate {background_rate_hz:.4g} Hz.")

Retained 218,870/230,400 cells; background rate 25.63 Hz.


## Prepare the FOV count estimate

Use the overlap between the spacecraft history and complete background time
bins. The time-resolved background estimates the fraction retained by a 60°
FOV selection. The smooth background remains the likelihood template.

In [30]:
def interval_integrals(edges, amounts, query):
    """Cumulative counts or livetime, with linear interpolation inside intervals."""
    return np.interp(query, edges, np.r_[0.0, np.cumsum(amounts)],
                     left=0.0, right=amounts.sum())


times = sc_orientation.obstime.unix
live = sc_orientation.livetime.to_value(u.s)
duration = np.diff(times)
if (np.any(duration <= 0) or np.any(np.diff(time_edges) <= 0)
        or np.any(live < 0) or np.any(live > duration + 1e-5)):
    raise ValueError("Invalid time edges or spacecraft livetimes")
complete_time_bins = (time_edges[:-1] >= times[0] - 1e-5) & (time_edges[1:] <= times[-1] + 1e-5)
covered = np.flatnonzero(complete_time_bins)
if not len(covered):
    raise ValueError("No complete background time bins overlap the spacecraft history")
common_start, common_end = time_edges[covered[0]], time_edges[covered[-1] + 1]
common_fraction = np.maximum(
    0, np.minimum(times[1:], common_end) - np.maximum(times[:-1], common_start),
) / duration
background_live = np.diff(interval_integrals(times, live, time_edges))
common_counts_em = counts_te[complete_time_bins].sum(axis=0)
# A whole energy bin may be masked out; give it zero weight in the FOV estimate.
smooth_em = background_values.sum(axis=(1, 2))
if np.any((smooth_em > 0) & (common_counts_em <= 0)):
    raise ValueError("Retained energy bins need background counts in the common interval")
if np.any(complete_time_bins & (background_live <= 0) & (counts_te.sum(axis=1) > 0)):
    raise ValueError("Background counts occur in a covered interval with no livetime")
smooth_common = smooth_em * np.divide(
    common_counts_em, counts_te.sum(axis=0),
    out=np.zeros_like(smooth_em), where=counts_te.sum(axis=0) > 0,
)
fov_coverage = dict(common_start_unix=common_start, common_end_unix=common_end,
                    common_livetime_s=np.sum(live * common_fraction))

For each source, accumulate small energy projections for the common interval
and its FOV-selected part. These are reused across cutoffs for the FOV gain
estimate. Earth occultation and the cell mask apply to both projections.
The Asimov source template itself is generated by `SourceInjector` at each cutoff.

In [31]:
def weighted_history(fraction):
    return SpacecraftHistory(
        sc_orientation.obstime, sc_orientation.attitude, sc_orientation.location,
        livetime=sc_orientation.livetime * fraction,
    )


def attitude_key(attitude):
    # Identical discretized attitudes have identical rotation vectors in each map.
    return tuple(attitude.as_rotvec().to_value(u.rad))


def prepare_fov_response(source, detector_response):
    """Build masked energy projections for the common interval and FOV selection."""
    coord = SkyCoord(source["ra_deg"], source["dec_deg"], unit="deg", frame="icrs")
    theta = 90.0 - sc_orientation.get_target_in_sc_frame(coord).lat.to_value(u.deg)[:-1]
    selected = theta <= FOV_MAX_OFFAXIS_DEG
    selected_live = np.diff(interval_integrals(times, live * selected, time_edges))
    fraction = np.divide(selected_live, background_live,
                         out=np.zeros_like(background_live), where=background_live > 0)
    background_cut = smooth_common * np.divide(
        (counts_te[complete_time_bins] * fraction[complete_time_bins, None]).sum(axis=0),
        common_counts_em, out=np.zeros_like(smooth_em), where=common_counts_em > 0,
    )

    histories = [sc_orientation, weighted_history(common_fraction),
                 weighted_history(common_fraction * selected)]
    maps = [history.get_scatt_map(nside=2 * background.axes["PsiChi"].nside,
                                  target_coord=coord, earth_occ=True)
            for history in histories]
    common_weights, cut_weights = [
        {attitude_key(att): weight.to_value(u.s)
         for att, weight in zip(mapping.attitudes, mapping.weights)}
        for mapping in maps[1:]
    ]
    data = EmCDSBinnedData(background)
    instrument = BinnedInstrumentResponse(detector_response, data)
    energy_axis = detector_response.axes["Ei"]
    common = np.zeros((energy_axis.nbins, len(smooth_em)))
    cut = np.zeros_like(common)
    area = np.zeros((energy_axis.nbins, *background_values.shape)) * u.cm**2
    for attitude in maps[0].attitudes:
        instrument.differential_effective_area(
            coord, energy_axis.centers, attitude=attitude, out=area,
        )
        retained_area = area.value * keep_cells[None]
        area_em = retained_area.sum(axis=(2, 3))
        key = attitude_key(attitude)
        common += area_em * common_weights.get(key, 0.0)
        cut += area_em * cut_weights.get(key, 0.0)
    return common, cut, background_cut

## Fit the source and background normalizations

SourceInjector caches its most recent point-source response for a fixed
sky position and spacecraft attitude/exposure map. Each cutoff spectrum is
folded into that response without repeating the detector-response rotation. `CachedTemplateLikelihood` stores
the resulting source and background templates and evaluates the Poisson
log-likelihood as their normalizations change:

$$\log L = \sum_{i:n_i>0} n_i\log\mu_i - \sum_i\mu_i,\qquad
\mu_i=aS_i+bB_i.$$

The source normalization is the spectral parameter `K`; the background
normalization is its event rate in Hz. All other source parameters stay fixed.
Only occupied cells need the logarithmic term; the expectation sum includes
**all retained cells**. The parameter-independent factorial term is omitted.

Create one threeML `JointLikelihood` per cutoff for the two hypothesis fits.
The optimizer uses the cached arrays, so amplitude changes do not repeat spectral
folding. Covariance calculations are disabled because this calculation needs TS.

In [32]:
class CachedTemplateLikelihood:
    """Poisson likelihood for fixed source/background shapes and free amplitudes."""

    def __init__(self, signal, background_counts, reference_K, bkg):
        self.signal = signal
        self.background = background_counts
        self.reference_K = reference_K
        self.bkg = bkg
        self.signal_total = signal.sum()
        self.background_total = background_counts.sum()
        self.reference_rate = self.background_total / livetime_s
        self.nobservations = len(signal)

    def set_model(self, model):
        # ThreeMLPluginInterface uses this method to supply the source model.
        source = next(iter(model.point_sources.values()))
        self.spectrum = source.spectrum.main.shape

    def set_counts(self, counts):
        occupied = counts > 0
        self.counts = counts[occupied]
        self.signal_observed = self.signal[occupied]
        self.background_observed = self.background[occupied]

    def get_log_like(self):
        source_scale = self.spectrum.K.value / self.reference_K
        background_scale = self.bkg.norm.to_value(u.Hz) / self.reference_rate
        expected = (source_scale * self.signal_observed
                    + background_scale * self.background_observed)
        if np.any(expected <= 0):
            return -np.inf
        return float(
            np.sum(self.counts * np.log(expected))
            - source_scale * self.signal_total
            - background_scale * self.background_total
        )


def make_threeML_fit(model, expected_source):
    source = next(iter(model.point_sources.values()))
    spectrum = source.spectrum.main.shape
    for parameter in source.parameters.values():
        parameter.free = False
    spectrum.K.remove_transformation()
    spectrum.K.min_value = 0.0
    spectrum.K.max_value = None
    spectrum.K.delta = 0.1 * spectrum.K.value
    spectrum.K.free = True

    bkg = FreeNormBinnedBackground({"Background": background},
                                  sc_history=sc_orientation, copy=True)
    likelihood = CachedTemplateLikelihood(expected_source, background_counts,
                                          spectrum.K.value, bkg)
    # The cached likelihood also receives the model through the response interface.
    plugin = ThreeMLPluginInterface("cosi", likelihood, response=likelihood, bkg=bkg)
    plugin.bkg_parameter["Background"] = Parameter(
        "Background", background_rate_hz, min_value=0, max_value=None,
        delta=0.05 * background_rate_hz, unit=u.Hz,
    )
    analysis = JointLikelihood(model, DataList(plugin), verbose=False)
    analysis.set_minimizer("minuit")
    return analysis, plugin, likelihood

For each Asimov dataset, fit source + background and then background alone with
`K = 0`. Compute `TS = 2 * (loglike_signal - loglike_null)`. Nonfinite or materially
negative TS values mark the fit as failed. Tiny negative values within
0.001 of zero are clipped to zero to allow for optimizer precision.

In [33]:
def run_threeML_fit(analysis):
    # Permit K = 0 when threeML clones the model to construct fit results.
    previous = astromodels_config.modeling.use_parameter_transforms
    try:
        astromodels_config.modeling.use_parameter_transforms = False
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="divide by zero encountered in _log10",
                                    category=RuntimeWarning)
            analysis.fit(quiet=True, compute_covariance=False, n_samples=1)
    finally:
        astromodels_config.modeling.use_parameter_transforms = previous


def fit_asimov_counts(counts, analysis, plugin, likelihood):
    spectrum = likelihood.spectrum
    reference_K = likelihood.reference_K
    likelihood.set_counts(counts)
    try:
        # Fit source + background, with both normalizations free.
        spectrum.K.value = reference_K
        spectrum.K.free = True
        plugin.bkg_parameter["Background"].value = background_rate_hz
        run_threeML_fit(analysis)
        loglike_signal = float(plugin.get_log_like())
        K_fit = spectrum.K.value
        background_fit = plugin.bkg_parameter["Background"].value

        # Fit the background-only hypothesis with source normalization fixed at zero.
        spectrum.K.value = 0.0
        spectrum.K.free = False
        run_threeML_fit(analysis)
        loglike_null = float(plugin.get_log_like())
        background_null = plugin.bkg_parameter["Background"].value
        ts_raw = 2 * (loglike_signal - loglike_null)
        if not np.isfinite(ts_raw) or ts_raw < -1e-3:
            raise ValueError(f"Invalid fitted TS: {ts_raw}")
        return dict(
            ts=max(0.0, ts_raw), loglike_signal=loglike_signal, loglike_null=loglike_null,
            K_fit_ph_cm2_s_keV=K_fit, background_rate_fit_hz=background_fit,
            background_rate_null_hz=background_null,
        )
    finally:
        spectrum.K.value = reference_K
        spectrum.K.free = True

## Summarize one cutoff

Record the fitted Asimov TS. A failed fit makes the cutoff inconclusive.
For a valid fit, estimate the FOV improvement as $G=f_S/\sqrt{f_B}$ and
$TS_{\mathrm{Asimov,FOV}}\approx G^2 TS_{\mathrm{Asimov}}$.
This is a count estimate rather than a FOV-selected likelihood refit.

In [34]:
def summarize_scenario(scenario, spectrum, expected_source, fov_counts, fit_result):
    fit_ok = fit_result["status"] == "ok"
    asimov_ts = fit_result["ts"] if fit_ok else np.nan
    common_signal = fov_counts["signal"].sum()
    source_fraction = fov_counts["signal_cut"].sum() / common_signal if common_signal > 0 else np.nan
    background_fraction = fov_counts["background_cut"].sum() / smooth_common.sum()
    gain = source_fraction / np.sqrt(background_fraction) if background_fraction > 0 else np.nan
    fov_ok = np.isfinite(gain) and gain > 0
    ts_fov = asimov_ts * gain**2 if fit_ok and fov_ok else np.nan
    photon_flux, energy_flux = band_fluxes(spectrum, ENERGY_BAND_KEV)

    result = dict(
        **scenario, K_ph_cm2_s_keV=spectrum.K.value,
        photon_flux_0p1_10_MeV_ph_cm2_s=photon_flux,
        energy_flux_0p1_10_MeV_erg_cm2_s=energy_flux,
        fit_status=fit_result["status"], fit_error=fit_result["error"],
        classification_status="complete" if fit_ok else "inconclusive_fit_failure",
        asimov_ts_3mo=asimov_ts,
        gain=gain, fov_status="ok" if fov_ok else "no usable selected exposure",
        signal_fraction=source_fraction, background_fraction=background_fraction,
        expected_source_counts_3mo=expected_source.sum(),
        asimov_ts_3mo_fov_est=ts_fov, asimov_ts_24mo_fov_est=EXPOSURE_SCALE_2YR * ts_fov,
    )
    energy_rows = []
    for i, (low, high) in enumerate(zip(measured_energy_edges[:-1], measured_energy_edges[1:])):
        energy_rows.append(dict(
            source=scenario["source"], cutoff_keV=scenario["cutoff_keV"],
            energy_min_keV=low, energy_max_keV=high,
            **{name: counts[i] for name, counts in fov_counts.items()},
        ))
    return result, energy_rows

## Build and fit the Asimov counts

Create a Galactic-coordinate point-source model and call
`injector.inject_model(..., fluctuate=False)`. Project its expected counts onto
`Em`, `Phi`, and `PsiChi`, then apply the background-cell mask. For each retained
cell, add the expected background counts:

$$n_{A,i}=S_i+B_i.$$

Keep the fractional counts without rounding. The dataset is deterministic and
contains no random draws. Fit source + background and background alone, varying
their normalizations while keeping template shapes fixed. Asimov TS estimates
median expected sensitivity; it does not measure the spread of individual datasets.

Each source is evaluated separately against the background. Scanned sources stop
at their first valid cutoff with three-month FOV-estimated Asimov TS ≥ 25.
Inspect `failed_fits` for unsuccessful fits.

In [35]:
result_rows, fit_rows, energy_rows, decisions = [], [], [], []
# Reuse this injector so consecutive cutoffs share its point-source response.
injector = SourceInjector(response_path=paths["response"], response_frame="spacecraftframe")

with FullDetectorResponse.open(paths["response"]) as detector_response:
    energy_axis = detector_response.axes["Ei"]
    measured_energy_edges = background.axes["Em"].edges.to_value(u.keV)

    for name, source_scenarios in scenarios.groupby("source", sort=False):
        print(f"{name}: preparing response...", end=" ", flush=True)
        common_response, cut_response, background_cut = prepare_fov_response(
            source_scenarios.iloc[0], detector_response,
        )
        source_results = []

        for scenario in source_scenarios.to_dict("records"):
            # Define this spectrum while preserving its BAT energy flux.
            spectrum = spectrum_at_cutoff(
                scenario["astromodel_index"], scenario["cutoff_keV"],
                scenario["bat_14_195_energy_flux_erg_cm2_s"],
            )
            flux = np.asarray(get_integrated_spectral_model(spectrum, energy_axis).contents, dtype=float)
            # Use Galactic coordinates so the injected sky cells match the background.
            coord = SkyCoord(scenario["ra_deg"], scenario["dec_deg"], unit="deg", frame="icrs").galactic
            source = PointSource(name, l=coord.l.deg, b=coord.b.deg, spectral_shape=spectrum)
            model = Model(source)
            injected_source = injector.inject_model(
                model=model, orientation=sc_orientation, fluctuate=False,
                make_spectrum_plot=False, make_PsiChi_plot=False, data_save_path=None,
            )
            signal = injected_source.project("Em", "Phi", "PsiChi").to_dense()
            for label in ("Em", "Phi"):
                np.testing.assert_array_equal(
                    signal.axes[label].edges.to_value(background.axes[label].unit),
                    background.axes[label].edges.value,
                )
            if signal.axes["PsiChi"] != background.axes["PsiChi"]:
                raise ValueError("Injected source and background sky bins differ")
            expected_source = np.asarray(signal.contents, dtype=float)[keep_cells]
            check_counts(expected_source, "Retained source expectation")
            fov_counts = dict(signal=flux @ common_response, signal_cut=flux @ cut_response,
                              background=smooth_common, background_cut=background_cut)

            # Fit the mean source + background counts, including fractional counts.
            asimov_counts = expected_source + background_counts
            analysis, plugin, likelihood = make_threeML_fit(model, expected_source)
            fit_result = dict(source=name, cutoff_keV=scenario["cutoff_keV"],
                              status="ok", error="", ts=np.nan)
            try:
                fit_result.update(fit_asimov_counts(asimov_counts, analysis, plugin, likelihood))
            except (ValueError, RuntimeError, FloatingPointError, FitFailed) as exc:
                fit_result.update(status="failed", error=f"{type(exc).__name__}: {exc}")
            fit_rows.append(fit_result)
            del analysis, plugin, likelihood

            result, energy_at_cutoff = summarize_scenario(
                scenario, spectrum, expected_source, fov_counts, fit_result,
            )
            result_rows.append(result)
            energy_rows.extend(energy_at_cutoff)
            source_results.append(result)
            scanned = scenario["scenario_type"] == "bat_cutoff_scan"
            if scanned and result["asimov_ts_3mo_fov_est"] >= TARGET_TS:
                break

        del common_response, cut_response
        source_results = pd.DataFrame(source_results)
        if (source_results["asimov_ts_3mo_fov_est"] >= TARGET_TS).any():
            status = "detected_3mo"
        elif (source_results["asimov_ts_24mo_fov_est"] >= TARGET_TS).any():
            status = "detected_24mo_only"
        elif source_results["asimov_ts_3mo_fov_est"].isna().any():
            status = "inconclusive"
        else:
            status = "below_target"
        decisions.append(dict(source=name, run_status=status,
                              n_scenarios_tested=len(source_results)))
        print(f"{len(source_results)} cutoff(s), {status}.", flush=True)

NGC_4151: preparing response... 2 cutoff(s), detected_3mo.
NGC_2110: preparing response... 1 cutoff(s), detected_3mo.
Circinus_Galaxy: preparing response... 10 cutoff(s), detected_24mo_only.
NGC_4945: preparing response... 3 cutoff(s), detected_3mo.
IC_4329A: preparing response... 10 cutoff(s), detected_24mo_only.
NGC_4388: preparing response... 1 cutoff(s), detected_24mo_only.
NGC_5506: preparing response... 1 cutoff(s), below_target.
MCG_05_23_016: preparing response... 1 cutoff(s), detected_24mo_only.
NGC_4507: preparing response... 1 cutoff(s), below_target.
NGC_7172: preparing response... 10 cutoff(s), detected_24mo_only.
NGC_3783: preparing response... 10 cutoff(s), detected_24mo_only.
MCG_08_11_011: preparing response... 10 cutoff(s), detected_24mo_only.
BAT_4U_1344_60: preparing response... 10 cutoff(s), below_target.
ESO_103_035: preparing response... 10 cutoff(s), below_target.
Mrk_926: preparing response... 10 cutoff(s), detected_24mo_only.
MR_2251_178: preparing response...

## Collect the results

Detailed fits, failed fits, FOV counts, scenario summaries, and source decisions
remain in memory. After displaying the final tables, save a single CSV containing
one row per source, including detectable, undetected, and inconclusive sources.

In [36]:
summary = pd.DataFrame(result_rows)
fits = pd.DataFrame(fit_rows)
failed_fits = fits.loc[fits["status"] != "ok"]
source_run_decisions = pd.DataFrame(decisions)
fov_energy_counts = pd.DataFrame(energy_rows)
fov_estimates = summary[["source", "cutoff_keV", "gain", "fov_status",
                         "signal_fraction", "background_fraction"]].copy()
print(f"Evaluated {len(fits):,}/{maximum_scenarios:,} possible Asimov datasets; "
      f"{len(failed_fits)} failed fits.")

Evaluated 205/220 possible Asimov datasets; 0 failed fits.


## Detectable and undetected sources

The detectable-source table is sorted by the lowest **evaluated** cutoff reaching
the approximate 24-month Asimov TS target, from low to high. The three-month crossing is
reported separately. The undetected-source table is sorted by its lowest tested
cutoff, also from low to high; it has no qualifying detection cutoff. Ties in
either table follow BAT catalog rank (brighter BAT flux first).

The Asimov TS columns report the highest valid TS among evaluated cutoffs. Fixed-cutoff
sources have only their published value. An empty crossing means no evaluated
scenario qualified.
The undetected table distinguishes conclusive non-detections from incomplete
fits or unusable FOV estimates.

Radio-quiet classifications are read from `RADIO_CLASS_PATH`, using
[Phillipson et al. (2023), Table 1](https://arxiv.org/abs/2211.13774).
Only classified radio-quiet sources appear in these tables. Photon index Γ is
reported as a positive number; measured-cutoff sources use the published index
and scanned sources use the BAT index. `balokovic_cutoff_keV` gives the published
central cutoff or limit (for example, `>240`); `unavailable` means no matching
entry in the literature table. These labels are retained even when
`USE_MEASURED_CUTOFFS = False`. The tested and detection cutoff columns describe
the sensitivity scan separately. The final CSV contains all rows from
both tables, including radio classifications, Asimov TS values, cutoff values,
and detection status. Values are saved at full precision. To open the saved
results later, use `final_results = pd.read_csv(CHECKPOINT_PATH)`.
The table cell reads the literature labels directly; it can be rerun on the
existing `summary` without rerunning the source fits.

The approximate 24-month TS is eight times the three-month TS, assuming steady
emission and comparable exposure/background. It is not a separate simulation.
Asimov results describe expected sensitivity; no Poisson scatter is simulated.

In [ ]:
# Load literature labels without rebuilding scenarios or rerunning fits.
_, published_cutoffs = read_balokovic_cutoffs(BALOKOVIC_PATH)

# Limit the tables to CSV-classified radio-quiet sources.
table_summary = summary.loc[summary["source"].map(radio_class).eq("radio quiet")].copy()
target_ts = TARGET_TS
exposure_scale = EXPOSURE_SCALE_2YR
valid = table_summary.loc[
    (table_summary["classification_status"] == "complete") & (table_summary["fov_status"] == "ok")
    & (table_summary["fit_status"] == "ok")
    & np.isfinite(table_summary["asimov_ts_3mo_fov_est"])
].copy()


def first_threshold_crossings(frame, scale):
    reached = frame.loc[scale * frame["asimov_ts_3mo_fov_est"] >= target_ts]
    first = reached.sort_values(["catalog_rank", "cutoff_keV"]).drop_duplicates("source")
    return first.sort_values(["cutoff_keV", "catalog_rank"])


detected_sources_3mo = first_threshold_crossings(valid, 1.0)
detected_sources_24mo = first_threshold_crossings(valid, exposure_scale)
best = valid.sort_values(["asimov_ts_3mo_fov_est", "cutoff_keV"], ascending=[False, True])
best = best.drop_duplicates("source").rename(columns={
    "cutoff_keV": "best_cutoff_keV",
})
cosi_ranked_sources = (
    table_summary[["catalog_rank", "source", "counterpart", "scenario_type", "astromodel_index", "bat_14_195_energy_flux_erg_cm2_s"]].drop_duplicates("source")
    .merge(best[["source", "best_cutoff_keV", "asimov_ts_3mo_fov_est"]], on="source", how="left")
    .sort_values(["asimov_ts_3mo_fov_est", "catalog_rank"], ascending=[False, True], na_position="last")
    .reset_index(drop=True)
)
cosi_ranked_sources["cosi_rank"] = pd.Series(
    np.arange(1, len(cosi_ranked_sources) + 1), dtype="Int64",
).where(cosi_ranked_sources["asimov_ts_3mo_fov_est"].notna())
cosi_ranked_sources["asimov_ts_24mo_fov_est"] = exposure_scale * cosi_ranked_sources["asimov_ts_3mo_fov_est"]
cosi_ranked_sources["photon_index"] = -cosi_ranked_sources["astromodel_index"]
cosi_ranked_sources["balokovic_cutoff_keV"] = (
    cosi_ranked_sources["counterpart"].map(source_key).map(published_cutoffs).fillna("unavailable")
)
cosi_ranked_sources["radio_class"] = cosi_ranked_sources["source"].map(radio_class).fillna("unclassified")
cosi_ranked_sources["is_radio_quiet"] = cosi_ranked_sources["radio_class"].map(
    {"radio quiet": True, "radio loud": False},
).astype("boolean")
for period, detected in (("3mo", detected_sources_3mo), ("24mo", detected_sources_24mo)):
    cosi_ranked_sources[f"lowest_tested_cutoff_{period}_keV"] = cosi_ranked_sources["source"].map(
        detected.set_index("source")["cutoff_keV"],
    )
lowest_cutoff_24mo = cosi_ranked_sources.loc[
    cosi_ranked_sources["lowest_tested_cutoff_24mo_keV"].notna()
].sort_values(["lowest_tested_cutoff_24mo_keV", "catalog_rank"])
# Use the 24-month estimate to define the detectable group; include 3-month crossings too.
detectable_sources = cosi_ranked_sources.loc[
    cosi_ranked_sources["source"].isin(detected_sources_24mo["source"])
].sort_values(["lowest_tested_cutoff_24mo_keV", "catalog_rank"]).reset_index(drop=True)
undetected_sources = cosi_ranked_sources.loc[
    ~cosi_ranked_sources["source"].isin(detected_sources_24mo["source"])
].copy()
# No qualifying cutoff exists for this group; show its lowest tested cutoff.
undetected_sources["lowest_tested_cutoff_keV"] = undetected_sources["source"].map(
    table_summary.groupby("source")["cutoff_keV"].min(),
)
undetected_sources = undetected_sources.sort_values(
    ["lowest_tested_cutoff_keV", "catalog_rank"], na_position="last",
).reset_index(drop=True)
incomplete = table_summary["asimov_ts_3mo_fov_est"].isna().groupby(table_summary["source"]).any()
undetected_sources["status"] = undetected_sources["source"].map(
    incomplete.map({True: "inconclusive", False: "below target at all tested cutoffs"}),
)
columns = [
    "source", "radio_class", "scenario_type", "photon_index", "balokovic_cutoff_keV",
    "bat_14_195_energy_flux_erg_cm2_s",
    "asimov_ts_3mo_fov_est", "asimov_ts_24mo_fov_est",
]
print(f"Asimov-detectable: {len(detected_sources_3mo)} at 3 months; "
      f"{len(detectable_sources)} at approximately 24 months (TS ≥ {target_ts:g}).")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.precision", 3):
    display(detectable_sources[columns + [
        "lowest_tested_cutoff_3mo_keV", "lowest_tested_cutoff_24mo_keV",
    ]])
print(f"Without an Asimov detection within the flux selection: {len(undetected_sources)} "
      "(inconclusive runs are labelled).")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.precision", 3):
    display(undetected_sources[columns + ["lowest_tested_cutoff_keV", "status"]])

In [ ]:
final_results = pd.concat([detectable_sources, undetected_sources], ignore_index=True)
final_results["lowest_tested_cutoff_keV"] = final_results["source"].map(
    summary.groupby("source")["cutoff_keV"].min(),
)
detected = final_results["status"].isna()
final_results.loc[detected, "status"] = np.where(
    final_results.loc[detected, "lowest_tested_cutoff_3mo_keV"].notna(),
    "detectable at 3 months", "detectable at 24 months only",
)
final_results = final_results[columns + [
    "lowest_tested_cutoff_3mo_keV", "lowest_tested_cutoff_24mo_keV",
    "lowest_tested_cutoff_keV", "status",
]]
final_results.to_csv(CHECKPOINT_PATH, index=False)
print(f"Saved {len(final_results)} sources to {CHECKPOINT_PATH}")

Saved 31 sources to /Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/Sensitivity_calculator/bat157_asimov_results.csv


## Radio-quiet sources with measured Baloković cutoffs below 100 keV

List the measured central cutoffs strictly below 100 keV for radio-quiet sources
within this analysis's BAT flux selection, sorted by decreasing BAT flux. Cutoff
lower limits and imposed scan cutoffs are excluded. Photon index Γ is the
published value used for that fixed-cutoff model. Detection labels use the same
Asimov TS threshold and FOV estimates at three and approximately 24 months;
failed fits or invalid FOV estimates are inconclusive.

This table can help select targets for subsequent non-thermal-component tests.

In [41]:
balokovic_below_100 = summary.loc[
    summary["scenario_type"].eq("balokovic_fixed")
    & summary["cutoff_keV"].lt(100.0)
    & summary["source"].map(radio_class).eq("radio quiet")
].copy()
balokovic_below_100["photon_index"] = -balokovic_below_100["astromodel_index"]
balokovic_below_100["radio_class"] = balokovic_below_100["source"].map(radio_class)
balokovic_below_100 = balokovic_below_100.rename(columns={"cutoff_keV": "balokovic_cutoff_keV"})
valid_cutoff_fit = (
    balokovic_below_100["classification_status"].eq("complete")
    & balokovic_below_100["fit_status"].eq("ok")
    & balokovic_below_100["fov_status"].eq("ok")
    & np.isfinite(balokovic_below_100["asimov_ts_3mo_fov_est"])
)
for period in ("3mo", "24mo"):
    balokovic_below_100[f"detection_{period}"] = np.where(
        valid_cutoff_fit,
        np.where(balokovic_below_100[f"asimov_ts_{period}_fov_est"] >= TARGET_TS,
                 "detected", "not detected"),
        "inconclusive",
    )
balokovic_below_100 = balokovic_below_100.sort_values(
    ["bat_14_195_energy_flux_erg_cm2_s", "catalog_rank"], ascending=[False, True],
).reset_index(drop=True)
print(f"Radio-quiet sources with measured Baloković cutoff < 100 keV: {len(balokovic_below_100)}")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.precision", 3):
    display(balokovic_below_100[[
        "source", "radio_class", "bat_14_195_energy_flux_erg_cm2_s", "photon_index",
        "balokovic_cutoff_keV", "asimov_ts_3mo_fov_est", "asimov_ts_24mo_fov_est",
        "detection_3mo", "detection_24mo",
    ]])

Radio-quiet sources with measured Baloković cutoff < 100 keV: 3


,source,radio_class,bat_14_195_energy_flux_erg_cm2_s,photon_index,balokovic_cutoff_keV,asimov_ts_3mo_fov_est,asimov_ts_24mo_fov_est,detection_3mo,detection_24mo
0,NGC_4507,radio quiet,1.964e-10,1.41,80.0,2.371,18.966,not detected,not detected
1,NGC_5728,radio quiet,9.253e-11,1.30,80.0,0.515,4.121,not detected,not detected
2,NGC_3281,radio quiet,8.804e-11,1.25,70.0,0.721,5.765,not detected,not detected
